# `=== Atelier Scikit-learn ===`

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Importation du dataset
df = pd.read_csv("../data/mesures_capteurs.csv")

# Exploration du dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


## `Partie 1 – Gestion des doublons`

### `1.1: Vérification de l’existence de doublons dans df`

In [10]:
print(f" Duplications: {df.duplicated().sum()}")

 Duplications: 5


### `1.2: Suppression des doublons`

In [13]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

### `1.3: gestion des Valeurs manquantes`

In [14]:
df.isna().sum()

id_mesure       0
date_heure      0
id_capteur      0
batiment        0
temperature     6
humidite        5
pression        5
consommation    5
etat            4
dtype: int64

In [15]:
# Statistiques descriptives
df.describe()

,temperature,humidite,pression,consommation
count,594.000000,595.000000,595.000000,595.000000
mean,24.896212,64.863395,1012.241059,208.693294
std,4.068454,10.760478,10.623413,72.320179
min,-18.500000,28.520000,850.000000,18.120000
25%,22.597500,58.090000,1006.890000,160.500000
50%,24.870000,65.370000,1012.960000,206.130000
75%,27.295000,71.590000,1017.835000,254.265000
max,58.700000,145.000000,1038.430000,875.000000


In [16]:
# Gestion des veleurs manquantes pour la variable numeriques
# colonne température
df['temperature'] = df['temperature'].fillna(df['temperature'].median())
# Colonne consommation
df['consommation'] = df['consommation'].fillna(df['consommation'].median())
# Colonne humidite
df['humidite'] = df['humidite'].fillna(df['humidite'].median())
# Colonne pression
df['pression'] = df['pression'].fillna(df['pression'].median())
df.info()

<class 'pandas.DataFrame'>
Index: 600 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     600 non-null    str    
 1   date_heure    600 non-null    str    
 2   id_capteur    600 non-null    str    
 3   batiment      600 non-null    str    
 4   temperature   600 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          596 non-null    str    
dtypes: float64(4), str(5)
memory usage: 46.9 KB


In [17]:
# Gestion des veleurs manquantes pour la variable numeriques
df["etat"].mode()

0    OK
Name: etat, dtype: str

In [18]:
df["etat"] = df["etat"].fillna("OK")
df.isnull().sum()

id_mesure       0
date_heure      0
id_capteur      0
batiment        0
temperature     0
humidite        0
pression        0
consommation    0
etat            0
dtype: int64

## `Partie 2 – Sélection de y (cible) et X (caractéristiques)`

### `2.1: On définit "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives`

`Selection des features`

In [25]:
X = df[['temperature', 'humidite', 'pression', 'consommation']]

`Selection des labels`

In [26]:
y = df[['etat']]


### `2.2: Affichage des cinq premières lignes de X et de y`

In [27]:
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [28]:
y.head()

,etat
0,OK
1,OK
2,OK
3,OK
4,OK


### `2.3: Quel est le type du problème de machine learning ?`

Le type de problème en question est un problème de classification, l'objectif est de classer les enreigistrements dans les classe: `OK`, `ALERTE` et `ERREUR` en fontion des valeurs de la temperature, de l'humidité, de la pression et de la consommation

## `Partie 3 – Découpage Train/Test`

#### `3.1:` Division de `X `en deux ensembles distincts : un pour l'entraînement `(train)` et un pour le test `(test)`. Avec les conditions suivantes : `20%` des données serviront au test ; garantir la `reproductibilité` du découpage ; conserver les mêmes `proportions de classes` dans l'ensemble de `train` et de `test` que dans les données d'origine`

In [33]:
from sklearn.model_selection import train_test_split

# Division en train et test des features et labels
X_train, X_test, y_train, y_test = train_test_split(
     X,                  # Features
     y,                  # Target
     test_size=0.2,      # 20% des données pour le test
     random_state=42,    # Garantir la reproductibilité du découpage
     stratify=y          # conserver les meme proportions de classe dans train et test
)

## `Partie 4 – Gestion des valeurs manquantes`

### `4.1: Vérification de l’existence de valeurs manquantes`

In [29]:
X.isnull().sum()

temperature     0
humidite        0
pression        0
consommation    0
dtype: int64

### `4.2: Sélection de SimpleImputer avec la médiane`

In [34]:
from sklearn.impute import SimpleImputer

# Instanciation de l'imputeur
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

### `4.3: Qu’est ce qui justifie le choix de la médiane ?`

Le dataset en question contient des outliers (valeurs abérantes) et cela influx sur le calcul de la moyenne et de la variance, et pas sur la mediane; ce qui justifie le choix de la médiane

### `4.4: Les paramètres (médianes) de l’imputeur sur X_train`

In [54]:
medianes = imputer.statistics_

for col, med in zip(X_train.columns, medianes):
     print(f"{col}  --- : {med:.4f}")

temperature  --- : 24.8700
humidite  --- : 65.3350
pression  --- : 1012.6700
consommation  --- : 206.1300


### `4.5: On déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test`

In [37]:
X_train_imputed

array([[  58.7 ,   81.51, 1026.79,  271.92],
       [  26.2 ,   76.27, 1016.74,  191.33],
       [  27.27,   75.68, 1013.78,  335.2 ],
       ...,
       [  21.64,   52.74, 1002.78,  157.77],
       [  26.27,   68.79, 1003.12,  244.9 ],
       [  32.72,   53.28, 1010.05,  195.37]], shape=(480, 4))

In [38]:
X_test_imputed

array([[  19.27,   58.39, 1003.74,  136.56],
       [  25.96,   62.79, 1012.17,  228.74],
       [  24.87,   69.97, 1014.14,  181.07],
       [  33.29,   50.43, 1009.87,  356.76],
       [  30.38,   65.98, 1005.48,  256.18],
       [  27.85,   58.01, 1012.29,  208.09],
       [  26.04,   67.17, 1011.38,  253.99],
       [  28.66,   35.83, 1022.52,  142.86],
       [  25.02,   79.27, 1027.7 ,  214.88],
       [  20.71,   72.01, 1002.15,  144.24],
       [  24.86,   68.77, 1003.98,  289.79],
       [  25.91,   49.2 , 1016.57,  201.37],
       [  29.5 ,   81.47, 1015.48,  256.53],
       [  27.33,   89.45, 1001.98,  346.24],
       [  22.96,   42.4 , 1011.47,  142.67],
       [  23.64,   48.79,  994.25,  222.42],
       [  20.72,   75.27,  997.63,  230.8 ],
       [  22.79,   68.34, 1013.41,  115.55],
       [  30.92,   67.19, 1009.71,  329.78],
       [  26.06,   77.13, 1021.87,  108.36],
       [  29.97,   68.65, 1007.19,  267.37],
       [  24.3 ,   58.75, 1023.6 ,  244.86],
       [  

## `Partie 5 – Mise à l'échelle`

### `5.1: Sélection de StandardScaler pour mettre à l’échelle les transformés de l’imputation`

In [40]:
from sklearn.preprocessing import StandardScaler

# Instanciation de l'objet scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
x_test_scaled = scaler.transform(X_test_imputed)

### `5.2: Qu’est ce qui justifie la standardisation ?`

On applique la `standardisation` pour que les valeurs des differentes variable partage une echèlle comparable, et comme notre dataset contient des outliers il est indispensable de mettre à l'échelle les valeur d'oú l'importance de la standardisation,. De plus certains algo sont sensibles à l'echelle des variables.

### `5.3: On donne les paramètres (moyennes et écart-types) du scaleur sur X_train_imputed`

`Moyennes`

In [60]:
means = scaler.mean_
print(f"Moyennes de chaque features:")
for col, med in zip(X_train.columns, means):
     print(f"{col}  --- : {med:.4f}")

Moyennes de chaque features:
temperature  --- : 24.9269
humidite  --- : 64.6353
pression  --- : 1011.9900
consommation  --- : 208.4475


`Ecart-types`

In [59]:
ecart_types = scaler.mean_
print(f"Ecart-types de chaque features:")
for col, med in zip(X_train.columns, ecart_types):
     print(f"{col}  --- : {med:.4f}")

Ecart-types de chaque features:
temperature  --- : 24.9269
humidite  --- : 64.6353
pression  --- : 1011.9900
consommation  --- : 208.4475


### `5.4: On déterminer X_train_scaled et X_test_scaled, les transformés de X_train_imputed et X_test_imputed`

In [64]:
print(X_train_scaled)

[[ 8.16891196  1.67353001  1.35520087  0.85676261]
 [ 0.30793852  1.15385897  0.43494622 -0.23105564]
 [ 0.56674595  1.09534639  0.16390605  1.71092738]
 ...
 [-0.79501653 -1.17970203 -0.84333784 -0.68405453]
 [ 0.32486984  0.41203847 -0.81220485  0.49204181]
 [ 1.88497073 -1.12614814 -0.17764119 -0.176523  ]]


In [63]:
print(x_test_scaled)

[[-1.36826290e+00 -6.19369700e-01 -7.55432917e-01 -9.70350911e-01]
 [ 2.49888252e-01 -1.83004703e-01  1.64821727e-02  2.73911236e-01]
 [-1.37567035e-02  5.29063633e-01  1.96870396e-01 -3.69546960e-01]
 [ 2.02284011e+00 -1.40879365e+00 -1.94123368e-01  2.00194812e+00]
 [ 1.31898064e+00  1.33359920e-01 -5.96105247e-01  6.44301270e-01]
 [ 7.07034092e-01 -6.57055768e-01  2.74702879e-02 -4.82616144e-03]
 [ 2.69238340e-01  2.51376817e-01 -5.58562520e-02  6.14740258e-01]
 [ 9.02953738e-01 -2.85673205e+00  9.64207105e-01 -8.85312382e-01]
 [ 2.25247124e-02  1.45138056e+00  1.43852741e+00  8.68264745e-02]
 [-1.01996130e+00  7.31378314e-01 -9.01025442e-01 -8.66684895e-01]
 [-1.61754646e-02  4.10054998e-01 -7.33456686e-01  1.09797507e+00]
 [ 2.37794447e-01 -1.53077750e+00  4.19379728e-01 -9.55339248e-02]
 [ 1.10612967e+00  1.66956306e+00  3.19571016e-01  6.49025633e-01]
 [ 5.81258517e-01  2.46097048e+00 -9.16591939e-01  1.85994728e+00]
 [-4.75740066e-01 -2.20515977e+00 -4.76151657e-02 -8.87877036e

## `Partie 6: Ecodage de la Target`

In [79]:

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(categories=[['OK', 'ALERTE', 'ERREUR']])

y_train_encoded = encoder.fit_transform(y_train).ravel()
y_test_encoded = encoder.transform(y_test).ravel()

""" 
Encodage avec pandas
mapping_cible = {'OK': 0, 'ALERTE': 1, 'ERREUR': 2}

# Appliquer sur votre variable cible
y_train_encoded = y_train['etat'].map(mapping_cible)
y_test_encoded = y_test['etat'].map(mapping_cible) """

y_train_encoded


array([2., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 2., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0.,
       1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 2., 0., 0.

## `Partie 7 – Entrainement et prédiction d’un modèle`

### `7.1: Sélection du modèle KNN (k plus proches voisins) avec 5, le nombre de voisins à prendre en compte`

In [83]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

### `7.2: Entrainement du modèle`

In [84]:
import time

temps_debut = time.time()
print(f"Début de l'entraînement : {time.strftime('%H:%M:%S', time.localtime(temps_debut))}")

knn.fit(X_train_scaled, y_train_encoded)

temps_fin = time.time()
print(f"Fin de l'entraînement   : {time.strftime('%H:%M:%S', time.localtime(temps_fin))}")

# 5. Calculer et afficher la durée totale
print(f"Durée totale de l'entraînement : { temps_fin - temps_debut:.4f} secondes")


Début de l'entraînement : 15:08:37
Fin de l'entraînement   : 15:08:37
Durée totale de l'entraînement : 0.0047 secondes


### `7.3: On détermine y_pred, la prédiction du modèle avec l’ensemble de test`

In [100]:
from sklearn.metrics import classification_report, accuracy_score

y_pred_encoded = knn.predict(x_test_scaled)

### `7.4: Affichage de quelques prédictions`

In [101]:
y_pred_texte = encoder.inverse_transform(y_pred_encoded.reshape(-1, 1)).ravel()
y_test_texte = encoder.inverse_transform(y_test_encoded.reshape(-1, 1)).ravel()

In [102]:
print(y_pred_texte[:5])

['OK' 'OK' 'OK' 'ALERTE' 'OK']


### `7.5: Comparaison des prédictions avec les vraies valeurs`

In [104]:
exactitude = accuracy_score(y_test_texte, y_pred_texte)
print(f"Taux de prédictions correctes : {exactitude * 100:.2f}%\n")

# 3. Boucle pour afficher les 10 premières valeurs
print(f"{'Index':<6} | {'Valeur Réelle':<15} | {'Valeur Prédite':<15} | Statut")
print("-" * 55)

for i in range(10):
    reelle = y_test_texte[i]
    predite = y_pred_texte[i]
    statut = "✅ Correct" if reelle == predite else "❌ Erreur"
    
    print(f"{i:<6} | {reelle:<15} | {predite:<15} | {statut}")

Taux de prédictions correctes : 96.67%

Index  | Valeur Réelle   | Valeur Prédite  | Statut
-------------------------------------------------------
0      | OK              | OK              | ✅ Correct
1      | OK              | OK              | ✅ Correct
2      | OK              | OK              | ✅ Correct
3      | ALERTE          | ALERTE          | ✅ Correct
4      | OK              | OK              | ✅ Correct
5      | OK              | OK              | ✅ Correct
6      | OK              | OK              | ✅ Correct
7      | OK              | OK              | ✅ Correct
8      | OK              | OK              | ✅ Correct
9      | OK              | OK              | ✅ Correct
